# LLM Preference Elicitation: Learning Internal Representations

## 1. Introduction & Methodology

In the previous phase, we reproduced core GRUM (Generalized Rank-Utility Model) results using standard social choice datasets. We now extend this to **LLM Preference Elicitation**, treating the LLM as an agent with internal preferences driven by input prompts.

### Agent Representation: The Embedding Choice
We represent the LLM's "state" for a given query as a feature vector $x_n$, derived using two distinct strategies:
- **Hidden States (HS)**: PCA-reduced activations from the LLM's last layer. Captures the model's internal latent state.
- **Sentence Transformers (ST)**: PCA-reduced semantic embeddings of the input prompt text. Captures external semantic meaning.

### Domain: Colors
We use the **Colors** domain (Blue, Red, Green, Purple, Yellow) to validate the elicitation process. This allows us to assess how well GRUM identifies intrinsic preferences ($\delta$) and personalized interaction patterns ($B$).

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

# Global Configuration
color_names = ["blue", "red", "green", "purple", "yellow"]
color_palette = {"blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c", "purple": "#9467bd", "yellow": "#d6d627"}

## 2. Load Data & Convergence Analysis
We load experiment results and measure how quickly the model parameters ($\delta$ and $B$) converge to their final values over the elicitation steps.

In [2]:
EXP_DIR = ROOT / "results" / "llm" / "llm_colors-20260326-114531"
output_dir = EXP_DIR / "outputs"

results_df_list = []
convergence_data = []
raw_results = []

if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            res = json.load(j)
            
            criterion = res.get("criterion", "social").capitalize()
            embedding = res.get("embedding_method", "")
            if not embedding: embedding = "HS" if "hidden_state" in f.name else "ST"
            else: embedding = "HS" if "hidden_state" in embedding else "ST"
            
            model_full = res.get("model_id", "").split("/")[-1]
            model_type = "Instruct" if "Instruct" in model_full else "Pretrained"
            
            history = res.get("history", {})
            if not history: continue
            
            steps = sorted(map(int, history.keys()))
            last_step = str(steps[-1])
            final_params = history[last_step]["grum"]
            final_delta = np.array(final_params["delta"])
            final_B = np.array(final_params["interaction"])
            
            raw_results.append({"Model": model_type, "Embedding": embedding, "Criterion": criterion, "delta": final_delta, "B": final_B})
            
            for i, color in enumerate(color_names):
                results_df_list.append({"Model Type": model_type, "Embedding": embedding, "Criterion": criterion, "Color": color, "Score (Delta)": final_delta[i]})
            
            # Convergence metrics over steps
            for s in steps:
                curr = history[str(s)]["grum"]
                d_mse = np.mean((np.array(curr["delta"]) - final_delta)**2)
                b_mse = np.mean((np.array(curr["interaction"]) - final_B)**2)
                convergence_data.append({"Step": s, "Embedding": embedding, "Model": model_type, "Delta MSE": d_mse, "B MSE": b_mse})

df = pd.DataFrame(results_df_list)
conv_df = pd.DataFrame(convergence_data)
print(f"Loaded {len(raw_results)} runs.")

### 2.1 Convergence Rate
How fast does the model settle on its final rankings?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=conv_df, x="Step", y="Delta MSE", hue="Embedding", ax=axes[0])
axes[0].set_title("Convergence of Delta (Intrinsic Preferences)")
axes[0].set_yscale('log')
sns.lineplot(data=conv_df, x="Step", y="B MSE", hue="Embedding", ax=axes[1])
axes[1].set_title("Convergence of B (Interaction Matrix)")
axes[1].set_yscale('log')
plt.tight_layout()
plt.show()

## 3. Interaction Matrix ($B$) Analysis

### 3.1 Column Similarity Metric
High column similarity in $B$ indicates that the interaction term is absorbing a **global agent-level bias** (redundant across items) rather than learning true personalization.

In [ ]:
sim_results = []
for r in raw_results:
    B = r['B']
    col_sim = cosine_similarity(B.T)
    avg_col_sim = (np.sum(col_sim) - B.shape[1]) / (B.shape[1] * (B.shape[1] - 1))
    sim_results.append({"Model": r['Model'], "Embedding": r['Embedding'], "Avg Col Sim": avg_col_sim})

display(pd.DataFrame(sim_results).groupby(["Model", "Embedding"])[["Avg Col Sim"]].mean().round(4))

### 3.2 Visual Comparison: Interaction Patterns
Comparing $B$ heatmaps for both **Pretrained** and **Instruct** models. Note: Redundant columns (vertical stripes) appearing in current data suggest that $B$ needs a centering constraint across columns to be truly identifiable.

In [ ]:
def plot_b_matrix(B, title, ax):
    vmax = max(0.5, np.abs(B).max())
    sns.heatmap(B, annot=False, cmap="RdBu_r", center=0, ax=ax, vmax=vmax, vmin=-vmax,
                xticklabels=color_names, yticklabels=[f"F{i}" for i in range(B.shape[0])])
    ax.set_title(title)

for m_type in ["Pretrained", "Instruct"]:
    filtered = [r for r in raw_results if r['Model'] == m_type]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for j, emb in enumerate(["HS", "ST"]):
        m = [r for r in filtered if r['Embedding'] == emb and r['Criterion'] == 'Social']
        if m: plot_b_matrix(m[0]['B'], f"{m_type} | {emb} (Social)", axes[j])
        else: axes[j].axis('off')
    plt.suptitle(f"Interaction Matrix Comparison: {m_type} Model", fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

## 4. Preference Comparisons
Visualizing the intrinsic $\delta$ parameters across all dimensions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Model Type", ax=axes[0], errorbar=None, palette="Set1")
axes[0].set_title("Model Type Comparison")
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Embedding", ax=axes[1], errorbar=None, palette="Set2")
axes[1].set_title("Embedding Comparison")
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Criterion", ax=axes[2], errorbar=None, palette="Set3")
axes[2].set_title("Criterion Comparison")
for ax in axes: ax.axhline(0, color='black', alpha=0.3)
plt.tight_layout()
plt.show()

**Conclusions**
The model shows robust convergence of intrinsic preferences within ~100-200 steps. The extremely high similarity in $B$ columns (approaching 1.0) confirms that without additional constraints, the interaction term absorbs a global bias component. We implemented a centering constraint in the inference engine to resolve this in future runs, which will force $B$ to focus on the relative personalized variance.